In [ ]:
import os
from pathlib import Path
import shutil

import teehr

teehr.__version__

### Setup

In [ ]:
test_eval_dir = Path(Path().cwd(), "FIRO_test_evaluation")

# spark configuration
from teehr.evaluation.spark_session_utils import create_spark_session
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=2,
    executor_memory="50g",
    executor_cores=7
)

# access local evaluation
ev = teehr.LocalReadWriteEvaluation(
    dir_path=test_eval_dir,
    spark=spark
)

ev.enable_logging()

### Calculate normals and generate reference forecast

In [ ]:
# create annual hourly normals from observed data

ts_normals = teehr.SignatureTimeseriesGenerators.Normals()
ts_normals.temporal_resolution = "hour_of_year"
ts_normals.summary_statistic = "mean"
ev.generate.signature_timeseries(
    method=ts_normals,
    input_table_name="primary_timeseries",
    start_datetime="1999-10-01 00:00:00",
    end_datetime="2005-09-30 23:00:00",
    timestep="1 hour",
    fillna=True,
    dropna=False,
    update_variable_table=True
).write()

In [ ]:
# calculate hourly reference forecast

ref_fcst = teehr.BenchmarkForecastGenerators.ReferenceForecast()
reference_table_name = "primary_timeseries"
reference_table_filters = [
    "configuration_name = 'usgs_observations'",
    "variable_name = 'streamflow_hour_of_year_mean'",
    "unit_name = 'm^3/s'"
]
template_table_name = "secondary_timeseries"
template_table_filters = [
    "configuration_name = 'hefs_streamflow_forecast'",
    "variable_name = 'streamflow_hourly_inst'",
    "unit_name = 'm^3/s'",
    # "member = '1981'"
]
ev.configurations.add(
    teehr.Configuration(
        name="benchmark_forecast_hourly_normals",
        timeseries_type="secondary",
        description="Reference forecast based on USGS climatology"
    )
)

# TIP: ensure the reference table and template table have overlapping value_time
ev.generate.benchmark_forecast(
    method=ref_fcst,
    reference_table_name=reference_table_name,
    template_table_name=template_table_name,
    reference_table_filters=reference_table_filters,
    template_table_filters=template_table_filters,
    output_configuration_name="benchmark_forecast_hourly_normals",
).write(
    destination_table="secondary_timeseries"
)

### Kill Spark

In [ ]:
ev.spark.stop()